# Hypothesis Testing

In [3]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

import matplotlib.pylab as plt
%matplotlib inline

from scipy import special

from scipy import stats

## Ex 1:  Swain vs. Alabama ##

In [15]:
k = np.arange(101)
p = special.comb(100, k)*(0.26**k)*(0.74**(100-k))
#p = stats.binom.pmf(k, 100, .26),

fig, ax = plt.subplots()

ax.bar(k, p, width=1, ec='white');
ax.set_axisbelow(True)
ax.grid(alpha=0.25)
plt.xlim(1,60)
plt.title("Test Statistic Under Null Hypothesis: X~Bin(n=100, p=0.26)");
plt.xlabel("Number of Black People on Panel of 100")
plt.ylabel("Percent per unit")
plt.scatter(8, -.0001, color="red", s=30, label="Observed Test Statistic")
plt.legend()

In [ ]:
#Hypothesis Test:  
# Null:  p=0.26
# Alternative  p<0.26


# (1) Calculate the p-value: probability of region beyond observed value
print("P(X<=8) = ",p[0:9].sum())

# Or can do this directly using CDF:
from scipy import stats
stats.binom.cdf(8, 100, .26)
               


# (2) See if this is less than your pre-chosen significance level

**Conclusion:**

In [ ]:
#Conclusion:  since p<0.01, we reject the null hypothesis at the 1% significance level

## Ex 2: Alameda County Jury Panels ##

In [32]:
eth =['Asian', 'Black', 'Latino', 'White', 'Other']

jury = pd.DataFrame(
    {       
        'Eligible': [0.15, 0.18, 0.12, 0.54, 0.01],
         'Panels': [0.26, 0.08, 0.08, 0.54, 0.04]
        
    }, index = eth
)

jury

,Eligible,Panels
Asian,0.15,0.26
Black,0.18,0.08
Latino,0.12,0.08
White,0.54,0.54
Other,0.01,0.04


In [36]:
jury.plot.barh();


In [26]:
#Make an array with the eligible proportions assumed under the null hypothesis

null_hyp=np.array([0.15, 0.18, 0.12, 0.54, 0.01])



In [ ]:
# Simulate a draw of size N assuming the null hypothesis:


N=1453  #from the ACLU report (p. 3 table, TOTALS row)
simulated = stats.multinomial.rvs(N, null_hyp)

simulated_proportions = simulated/N

print(simulated)
print(simulated_proportions)

In [ ]:
# The actual observed distribution (Panels) looks quite different
# from the simulation -- try running this several times to confirm!
jury_with_simulated=jury.copy()
jury_with_simulated["Simulated"]=simulated_proportions

print(jury_with_simulated)

jury_with_simulated.plot.barh()

## Distance Between Distributions

In [ ]:

# In this case, we need to understand how each of the 5 categories
# differ from their expected values according to the model

jury_with_difference=jury.copy()
jury_with_difference["Difference"]=jury["Panels"]-jury["Eligible"]
jury_with_difference

## Total Variation Distance

In [ ]:
# This was the statistic we chose
def tvd(dist1, dist2):
    return sum(abs(dist1 - dist2))/2

In [ ]:
# The TVD of our observed data (Panels) from their expected values
# assuming the model is true 
obsvd_tvd = tvd(jury['Panels'],null_hyp)
obsvd_tvd

In [ ]:
# The TVD of a model simluation from its expected values
tvd(stats.multinomial.rvs(N, null_hyp)/N, null_hyp)

In [ ]:
def simulated_tvd():
    return tvd(stats.multinomial.rvs(N, null_hyp)/N, null_hyp)


num_simulations = 1000000

tvds = np.array([tvd(stats.multinomial.rvs(N, null_hyp)/N, null_hyp) for i in range(num_simulations)])


plt.hist(tvds, density=True);  
plt.xlabel("Total Distance Variation")
plt.ylabel("Percent per unit")
    

In [ ]:
# Add the observed test statistic to the plot:
plt.hist(tvds, density=True);  
plt.xlabel("Total Distance Variation")
plt.ylabel("Percent per unit")

    
plt.scatter(obsvd_tvd, -.001, color="red", s=40)

plt.show()



In [ ]:

# (1) Calculate the empirical p-value: simulation area beyond observed value

sum(tvds>=obsvd_tvd)/1000000

# (2) See if this is less than 1% (or our initially chosen significance level)

**Conclusion**:
Our empiricial p-value is $0$ (note that the actual theoretical p-value isn't 0, but since our simulations showed 0 observations out of $1000000$ that had a TVD greater than or equal to 0.14, we know that the theoretical p-value is at least less than $1/1000000$.

Given that our empirical p-value is less than our significance cutoff of $0.01$ we **REJECT THE NULL** hypothesis.      It is important for us to be precise about what we can conclude from our analysis:

**We can conclude that the distribution provided for the panelists who reported for service does not look like a random sample from the estimated distribution in the eligible population.**

Our analysis supports the ACLU’s conclusion that the panels were not representative of the distribution provided for the eligible jurors.

As with most such analyses, ours doesn’t say why the distributions are different.   See here for a more detailed discussion about this case:  https://inferentialthinking.com/chapters/11/2/Multiple_Categories.html

## Ex 3: Mendel and Pea Flowers ##

Mendel's null: 75% chance of purple flowers

Observed data:  grew 929 plants, of which 705 had purple flowers (and 224 white; Mendel 1866, Expt. 3)


In [ ]:
#Simulate one value of the test statistic under the null hypothesis:  75% chance of purple flowers out of sample of 929

predicted_proportions = np.array([.75, .25])

def test_statistic_purple_flowers(n=929):
    num_purple=stats.binom.rvs(n, 0.75)
    #print("Number of purple flowers out of 929:",num_purple)
    return abs(num_purple/n*100-75)

test_statistic_purple_flowers()

In [ ]:
# Simulate 10,000 copies of the test statistic
num_simulations = 10000;

outcomes=[]

for i in np.arange(num_simulations):
    outcomes = np.append(outcomes, test_statistic_purple_flowers(929))

plt.hist(outcomes, density=True);

#ax.set_axisbelow(True)
#ax.grid(alpha=0.25)
plt.xlabel("Abs(75-% Purple) in sample of 929 if the null true")
plt.ylabel("Percent per unit")
plt.show()


## Our Decision

In [ ]:
#Compare to observed test statistic:
## Mendel had 929 plants, of which 705 had purple flowers

observed_purples = 705 / 929*100

observed_test_statistic=abs(observed_purples-75)
observed_test_statistic


In [ ]:
#Plot

plt.hist(outcomes, density=True);
#ax.set_axisbelow(True)
#ax.grid(alpha=0.25)
plt.xlabel("Abs(75-% Purple) in sample of 929 if the null true")
plt.ylabel("Percent per unit")

plt.scatter(observed_test_statistic, -.0001, color="red", label = 'observed test statistic')
plt.legend()

plt.show()

In [ ]:
# (1) Calculate the EMPIRICAL p-value: simulation area beyond observed value

sum(outcomes>=observed_test_statistic)/ num_simulations

# (2) See if this is less than 5% (or our initially chosen significance level)


## Conclusion:  
Since the empirical p-value (about 0.54)> 0.05, we **FAIL TO REJECT** the null hypothesis that the proportion of purple flowers is 0.75.  The evidence is consistent with our null hypothesis.  This does NOT prove that our null is true.  It just proves that we don't have evidence to reject it, so we will continue using the null hypothesis.


## Ex 4: The TA's Defense ##

In [ ]:
scores = pd.read_csv('scores_by_section.csv')
scores

In [ ]:
scores.groupby("Section").count()

In [ ]:
scores.groupby("Section").mean()

In [ ]:
observed_average = 13.6667 

In [ ]:
scores["Midterm"].mean()

In [ ]:
random_sample=scores.sample(27,replace=False)
random_sample["Midterm"].mean()

In [ ]:
random_sample['Midterm'].mean()

In [ ]:
# Simulate one value of the test statistic 
# under the hypothesis that the section is like a random sample from the class

def random_sample_midterm_avg():
    random_sample = scores.sample(27, replace=False)
    return random_sample['Midterm'].mean()

In [ ]:
# Simulate 100,000 copies of the test statistic


num_sim=100000

sample_averages = np.array([random_sample_midterm_avg() for i in range(num_sim)])




In [ ]:
# Compare the simulated distribution of the statistic
# and the actual observed statistic

plt.hist(sample_averages, density=True);

plots.scatter(observed_average, -0.01, color='red', s=120);

## Calculating p-value

In [ ]:
# (1) Calculate the p-value: simulation area beyond observed value
np.count_nonzero(sample_averages <= observed_average) / num_sim
# (2) See if this is less than 5%

## Conclusion:  
Since 0.05746> 0.05, we **FAIL TO REJECT** the null hypothesis that any difference in scores for section 3 was just due to random chance

In [ ]:
from scipy import stats


## Theoretical Calculation:

We could have done this without using simulations thanks to the Central Limit Theorem!   

By the CLT, $$\bar{X}_{27} \sim N(\mu, \sigma^2/27)$$ where $\mu = $ pop mean = `scores["Midterm"].mean()`   and                    

$\sigma = $ pop_std = `pop_std=scores["Midterm"].std(ddof=0)`


Thus 

p-value =   $P(\bar{X}_{27}<=13.667)$ = `stats.norm.cdf(13.667,pop_mean, pop_std/np.sqrt(27))`


In [ ]:
pop_mean=scores["Midterm"].mean()

In [ ]:
pop_std=scores["Midterm"].std(ddof=0)

In [ ]:
stats.norm.cdf(13.667,pop_mean, pop_std/np.sqrt(27))

Again, since $p= 0.059> 0.05$, we **FAIL TO REJECT** the null hypothesis that any difference in scores for section 3 was just due to random chance

## Ex 5: Confidence Intervals as Hypothesis Tests ##

We can test

* **Null:** population proportion of purple flowers $= 0.75$
* **Alternative:** population proportion $\neq 0.75$
* **Significance level:** 5%

by building a **95% confidence interval** for the population proportion and checking whether the null value $0.75$ is inside it:

* null value **outside** the CI → reject the null
* null value **inside** the CI → fail to reject the null

The three simulations below show *why* this works, using Mendel's data (929 plants, 705 purple).


In [ ]:
n = 929                  # number of plants
p_null = 0.75            # null value
num_purple = 705         # Mendel's observed count

p_hat = num_purple / n                     # sample proportion
se = np.sqrt(p_hat * (1 - p_hat) / n)      # standard error (estimated from the sample)
margin = 1.96 * se                         # margin of error for a 95% CI

ci_low, ci_high = p_hat - margin, p_hat + margin

print("Sample proportion:", round(p_hat, 4))
print("95% CI: [", round(ci_low, 3), ",", round(ci_high, 3), "]")
print("Is the null value 0.75 inside the CI?", ci_low <= p_null <= ci_high)


### Simulation 1: When the null is true, about 5% of CIs miss the null value ###

Pretend the null is exactly true (the chance of purple really is 0.75). Draw many samples of 929 plants, and build a 95% CI from each one.

About 95% of the intervals contain 0.75, and about **5% miss it**. Each interval that misses is a sample where a 5%-level test would (wrongly) reject the null. So "the CI misses the null value" is the same rare event as "p-value ≤ 0.05."


In [ ]:
def simulated_ci(n=929, p=0.75):
    # Draw one sample under the null and return its 95% CI
    p_hat_sim = stats.binom.rvs(n, p) / n
    se_sim = np.sqrt(p_hat_sim * (1 - p_hat_sim) / n)
    return p_hat_sim - 1.96 * se_sim, p_hat_sim + 1.96 * se_sim

# Draw 40 intervals and plot them
num_intervals = 40
fig, ax = plt.subplots(figsize=(10, 7))
for i in range(num_intervals):
    low, high = simulated_ci()
    misses = (p_null < low) or (p_null > high)
    color = "darkorange" if misses else "steelblue"
    ax.plot([low, high], [i, i], color=color, lw=3)
    ax.plot([(low + high) / 2], [i], "o", color=color, ms=4)

ax.axvline(p_null, color="black", ls="--", lw=2, label="true value = null value 0.75")
ax.set_yticks([])
ax.set_xlabel("Proportion of purple plants")
ax.set_title("40 simulated 95% CIs when the null is true\n(orange = misses 0.75)")
ax.legend()
plt.show()


In [ ]:
# Repeat many times: what fraction of CIs miss the null value?
num_sims = 10000
misses = 0
for i in range(num_sims):
    low, high = simulated_ci()
    if p_null < low or p_null > high:
        misses = misses + 1

print("Fraction of 95% CIs that miss 0.75:", misses / num_sims)
# This should be close to 0.05, the significance level


### Simulation 2: The test and the CI measure the same distance ###

* **Test view:** stand at the null value 0.75. Is the estimate more than $1.96$ SE away? (Is it in a 2.5% tail of the null distribution?)
* **CI view:** stand at the estimate 0.759. Is the null value more than $1.96$ SE away? (Is it outside the CI?)

The distance between 0.75 and 0.759 is the same in both directions, so the two views give the same answer.

*Technical note:* the test view uses the SE computed from the null value, and the CI uses the SE computed from the estimate. They're nearly equal here, which is why the test and the CI agree approximately (not always exactly) for proportions.


In [ ]:
# Simulate the distribution of the sample proportion under the null
num_sims = 100000
null_p_hats = stats.binom.rvs(n, p_null, size=num_sims) / n

# The middle 95% of the null distribution (the "fail to reject" region)
lower_cut = np.percentile(null_p_hats, 2.5)
upper_cut = np.percentile(null_p_hats, 97.5)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Test view
ax1.hist(null_p_hats, bins=40, density=True, color="steelblue", alpha=0.7)
ax1.axvline(lower_cut, color="darkorange", lw=2, label="2.5% cutoffs")
ax1.axvline(upper_cut, color="darkorange", lw=2)
ax1.axvline(p_hat, color="black", ls="--", lw=2, label="estimate 0.759")
ax1.set_title("Test view: null distribution of the sample proportion")
ax1.legend()

# CI view
ax2.plot([ci_low, ci_high], [0, 0], color="steelblue", lw=8, label="95% CI")
ax2.plot([p_hat], [0], "o", color="black", ms=10, label="estimate 0.759")
ax2.axvline(p_null, color="darkorange", ls="--", lw=2, label="null value 0.75")
ax2.set_yticks([])
ax2.set_title("CI view: 95% CI around the estimate")
ax2.set_xlabel("Proportion of purple plants")
ax2.legend()
plt.show()

print("Distance between estimate and null value:", round(abs(p_hat - p_null), 4))
print("Margin of error (1.96 SE):", round(margin, 4))
print("Estimate inside the middle 95% of the null distribution?", lower_cut <= p_hat <= upper_cut)
print("Null value inside the 95% CI?", ci_low <= p_null <= ci_high)


### Simulation 3: A CI is a test run on every possible null value ###

Keep Mendel's data fixed. For each possible null value, run the test and record its p-value. The null values with **p-value > 0.05** (the ones we would *not* reject) make up exactly the **95% CI**.

Here we use the normal approximation for each p-value: $z = \dfrac{\hat{p} - p_0}{SE}$, p-value $= 2 \cdot P(Z \geq |z|)$.


In [ ]:
null_values = np.linspace(0.715, 0.80, 5000)
z_scores = (p_hat - null_values) / se
p_values = 2 * stats.norm.sf(np.abs(z_scores))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(null_values, p_values, color="steelblue", lw=3)
ax.axhline(0.05, color="darkorange", ls="--", lw=2, label="significance level 0.05")
ax.axvspan(ci_low, ci_high, color="steelblue", alpha=0.15, label="95% CI")
ax.set_xlabel("Hypothesized proportion of purple plants (null value)")
ax.set_ylabel("p-value")
ax.set_title("p-value for every possible null value (Mendel's data)")
ax.legend()
plt.show()

# Which null values are NOT rejected at the 5% level?
not_rejected = null_values[p_values > 0.05]
print("Null values not rejected range from", round(not_rejected.min(), 3), "to", round(not_rejected.max(), 3))
print("95% CI:                              ", round(ci_low, 3), "to", round(ci_high, 3))

for p0 in [0.75, 0.79]:
    p_val = 2 * stats.norm.sf(abs(p_hat - p0) / se)
    print("Null value", p0, ": p-value =", round(p_val, 3), "| inside CI?", ci_low <= p0 <= ci_high)


**Conclusion:** The null value 0.75 is inside the 95% CI [0.731, 0.786], so we **FAIL TO REJECT** the null at the 5% level. This matches the simulation-based test in Ex 3 (p-value about 0.54).

The CI also tells us more than the test alone: every proportion from about 0.731 to 0.786 is consistent with Mendel's data, so the data doesn't *prove* the 3:1 ratio. It just doesn't give evidence against it.
